In [1]:
# 구글 드라이브연결
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/AI/Data Set/항공편 지연 예측"

Mounted at /content/drive
/content/drive/MyDrive/AI/Data Set/항공편 지연 예측


# **1. 라이브러리 로드 및 설정**
**먼저 범주형 데이터 처리에 강력한 CatBoost와 데이터 핸들링을 위한 필수 라이브러리를 임포트합니다.**

In [4]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [9]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

# 데이터 로드 (파일 경로를 환경에 맞게 수정하세요)
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')

# **2. 데이터 전처리 (Preprocessing)**
**항공 데이터 특성상 시간과 날짜 정보가 중요합니다. 결측치 처리와 파생 변수 생성을 진행합니다.**

In [14]:
def preprocess_data(df):
    # 1. 시간 데이터 분리 (예: 1530 -> 15시, 30분)
    # Estimated_Departure_Time 등이 숫자형일 경우 처리
    df['Dep_Hour'] = df['Estimated_Departure_Time'] // 100
    df['Dep_Minute'] = df['Estimated_Departure_Time'] % 100

    # 2. 결측치 채우기 (범주형은 'Unknown', 수치형은 -1 혹은 중앙값)
    # CatBoost는 자체적으로 결측치를 처리할 수 있으나, 명시적으로 'None' 처리가 유리할 때가 많습니다.
    cat_features = df.select_dtypes(include=['object']).columns.tolist()
    for col in cat_features:
        df[col] = df[col].fillna('None')

    return df

train_df = preprocess_data(train)
test_df = preprocess_data(test)

# 학습에 사용할 특성(Feature) 선택
# 'Delay'가 타겟 변수라면 제외
# Ensure 'Delay' column is properly handled for binary classification
# Filter out rows where 'Delay' might be 'None' (from original NaN) or other unexpected values
# and map 'Not_Delayed' to 0 and 'Delayed' to 1.
# This assumes 'None' is an invalid label for training and we want binary classification.
valid_labels = ['Not_Delayed', 'Delayed']
train_df_filtered = train_df[train_df['Delay'].isin(valid_labels)].copy()

X = train_df_filtered.drop(columns=['ID', 'Delay'])
y = train_df_filtered['Delay'].map({'Not_Delayed': 0, 'Delayed': 1})

# 범주형 변수 인덱스 추출 (CatBoost 핵심)
cat_features = X.select_dtypes(include=['object']).columns.tolist()

# **3. 모델 학습 (Training)**
**클래스 불균형이 있을 경우 class_weights를 조절하거나, 기본적인 하이퍼파라미터를 설정합니다.**

In [15]:
# 데이터 분할
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# CatBoost 모델 정의
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='Logloss', # eval_metric은 단일 값을 가져야 합니다.
    custom_metric=['F1'], # F1-Score를 모니터링하기 위해 custom_metric에 추가합니다.
    random_seed=42,
    verbose=100
)

# 학습
model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50
)

0:	learn: 0.6672272	test: 0.6672040	best: 0.6672040 (0)	total: 539ms	remaining: 8m 58s
100:	learn: 0.4443998	test: 0.4446592	best: 0.4446592 (100)	total: 58.3s	remaining: 8m 38s
200:	learn: 0.4405777	test: 0.4419714	best: 0.4419714 (200)	total: 1m 55s	remaining: 7m 40s
300:	learn: 0.4376327	test: 0.4403047	best: 0.4403047 (300)	total: 2m 52s	remaining: 6m 41s
400:	learn: 0.4351962	test: 0.4393276	best: 0.4393276 (400)	total: 3m 50s	remaining: 5m 43s
500:	learn: 0.4334060	test: 0.4387501	best: 0.4387423 (498)	total: 4m 44s	remaining: 4m 43s
600:	learn: 0.4318741	test: 0.4384358	best: 0.4384318 (596)	total: 5m 42s	remaining: 3m 47s
700:	learn: 0.4305421	test: 0.4382868	best: 0.4382783 (693)	total: 6m 39s	remaining: 2m 50s
800:	learn: 0.4292253	test: 0.4380342	best: 0.4380290 (797)	total: 7m 36s	remaining: 1m 53s
900:	learn: 0.4279693	test: 0.4379162	best: 0.4379157 (890)	total: 8m 36s	remaining: 56.7s
999:	learn: 0.4267702	test: 0.4378132	best: 0.4378132 (999)	total: 9m 33s	remaining: 0u

CatBoostClassifier(custom_metric=['F1'], depth=6, eval_metric='Logloss', iterations=1000, learning_rate=0.05, random_seed=42, verbose=100)

# **4. 평가 및 결과 제출**
**임계값(Threshold) 조정이 필요하다면 이 단계에서 수행합니다.**

In [16]:
# 검증 데이터 평가
val_preds = model.predict(X_val)
print(f"Validation F1 Score: {f1_score(y_val, val_preds, average='macro')}")
print(classification_report(y_val, val_preds))

# 테스트 데이터 예측 및 제출 파일 생성
test_preds = model.predict(test_df.drop(columns=['ID']))
submission = pd.DataFrame({
    'ID': test['ID'],
    'Delay': test_preds
})

submission.to_csv('dataset/baseline_submission.csv', index=False)

Validation F1 Score: 0.4647358061964815
              precision    recall  f1-score   support

           0       0.83      1.00      0.90     42001
           1       0.51      0.01      0.03      9000

    accuracy                           0.82     51001
   macro avg       0.67      0.51      0.46     51001
weighted avg       0.77      0.82      0.75     51001

